# Analysis template

Copy this notebook, rename it with your name, and build your charts below the last cell.

**Rules**
- Run cell 1 first. That's all the setup you need.
- Don't re-run the cleaning notebook; everything here reads the already-cleaned tables.
- `travel_times` is 349M rows. Filter on `year`, `month` or `mode` before anything else.
- Aggregate in Spark, then `toPandas()` only the small result.
- When you're done for the day, **delete the cluster** (don't just stop it).

In [ ]:
# 1. Setup + load every cleaned table
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
import matplotlib.pyplot as plt
import pandas as pd

spark = SparkSession.builder.getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Bluebikes timestamps are stored as TIMESTAMP, so this keeps every hour in Boston time.
spark.conf.set("spark.sql.session.timeZone", "America/New_York")
spark.conf.set("spark.sql.shuffle.partitions", "64")

B = "gs://ingestion-team3/data"
trips        = spark.read.parquet(f"{B}/cleaned_data/bluebikes/")
weather      = spark.read.parquet(f"{B}/cleaned_data/weather/")
alerts       = spark.read.parquet(f"{B}/clean/mbta/alerts")
travel_times = spark.read.parquet(f"{B}/clean/mbta/travel_times")

# Helper columns everyone ends up needing
trips = (trips
    .withColumn("start_hour", F.hour("started_at"))
    .withColumn("start_dow",  F.date_format("started_at", "E"))
    .withColumn("is_weekend", F.dayofweek("started_at").isin(1, 7))
    .withColumn("duration_min", F.col("duration_seconds") / 60))

for name, df in [("trips", trips), ("weather", weather),
                 ("alerts", alerts), ("travel_times", travel_times)]:
    df.createOrReplaceTempView(name)

print(f"trips        {trips.count():>12,}   (13,058,452)")
print(f"weather      {weather.count():>12,}   (1,096)")
print(f"alerts       {alerts.count():>12,}   (2,551,280)")
print(f"travel_times {travel_times.count():>12,}   (348,896,384)")

## What's in each table

| Table | One row is | Key columns | Join on |
|---|---|---|---|
| `trips` | a Bluebikes trip, 2023–2025 | `started_at`, `start_station_id/name`, `end_station_id`, `member_casual`, `duration_min`, `start_hour` | `trip_date` → weather; date + hour → alerts / travel times |
| `weather` | a day at Boston Logan | `weather_date`, `avg_temperature`, `precipitation`, `avg_wind_speed` | `weather_date` = `trip_date` |
| `alerts` | an alert × active period × route/stop | `effect_detail`, `severity_level`, `route_id`, `route_type`, `start_date`, `start_hour`, `active_period_start_dt/end_dt` | `start_date` + `start_hour` |
| `travel_times` | a train traversing a stop pair | `route_id`, `direction_id`, `from_stop_id`, `to_stop_id`, `travel_time_sec`, `service_date`, `departure_hour`, `mode` | `service_date` + `departure_hour` |

Notes: station-level work should use April 2023 onward (Jan–Mar 2023 uses legacy station IDs).
`route_type` 0/1 = subway, 3 = bus. Travel-time segments are *all* stop pairs, so compare a
segment to its own typical time rather than averaging raw seconds across segments.

In [ ]:
# 2. Quick look at any table — change the name and run
df = trips
df.printSchema()
df.show(5, truncate=False)

In [ ]:
# 3. Example join: daily trips by rider type, with that day's weather
daily = (trips.groupBy("trip_date", "member_casual").agg(F.count("*").alias("trips"))
              .join(weather, F.col("trip_date") == F.col("weather_date"), "left")
              .select("trip_date", "member_casual", "trips",
                      "avg_temperature", "precipitation"))
daily.orderBy("trip_date").show(6)

In [ ]:
# 4. Chart template: aggregate in Spark -> small pandas frame -> plot
result = (trips
    .groupBy("start_hour", "member_casual")
    .agg(F.count("*").alias("trips"))
    .orderBy("start_hour"))

result.show(48)                                  # keep a summary table: the rubric wants one
pdf = result.toPandas().pivot(index="start_hour", columns="member_casual", values="trips")

fig, ax = plt.subplots(figsize=(10, 5))
pdf.plot(ax=ax, marker="o")
ax.set_title("Trips by hour of day")              # every chart: title, axis labels, legend
ax.set_xlabel("Hour of day (Boston time)")
ax.set_ylabel("Trips")
ax.legend(title="Rider type")
fig.tight_layout()
plt.show()

## Your analysis

### Q?. *Your question, phrased as a question*
*One or two sentences on why it matters for station demand or stockout risk.*

*(code cells here)*

**Takeaway.** *1–3 sentences: what the chart shows and what it means.*